# 1+1 CDT spacetime visualization

This notebook generates an example visualization for a small 1+1-dimensional CDT run. It uses the Rust `cdt` binary as the simulation engine, reads the generated trace and summary artifacts, then renders the final exported spacetime triangulation.

Recent summaries include `final_triangulation.mesh`, a crate-owned JSON view of the final vertices and triangle index triples. The profile-based renderer remains as a fallback for older summaries.

## 1. Setup

From the repository root, run `just notebook-setup` once before opening this notebook. That installs the uv-managed notebook dependency group. The `justfile` contains the exact commands if you want to inspect what it does.

In [ ]:
from __future__ import annotations

import json
import math
import os
import shutil
import subprocess
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import polars as pl
from matplotlib.collections import LineCollection, PolyCollection

UTF8 = "utf-8"


@dataclass(frozen=True)
class RunPaths:
    output_dir: Path
    trace_csv: Path
    summary_json: Path
    figure_png: Path
    figure_svg: Path


@dataclass(frozen=True)
class Mesh:
    points: list[tuple[float, float]]
    times: list[int]
    triangles: list[tuple[int, int, int, str]]
    spacelike_edges: list[tuple[int, int]]
    timelike_edges: list[tuple[int, int]]


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "Cargo.toml").is_file() and (candidate / "pyproject.toml").is_file():
            return candidate
    message = "Run this notebook from inside the causal-triangulations repository."
    raise RuntimeError(message)


def run_command(command: list[str], *, cwd: Path, timeout: int = 180) -> subprocess.CompletedProcess[str]:
    result = subprocess.run(  # noqa: S603 - this notebook intentionally wraps the repository binary with fixed argv lists.
        command,
        cwd=cwd,
        text=True,
        capture_output=True,
        timeout=timeout,
        check=False,
    )
    if result.returncode != 0:
        command_text = " ".join(command)
        raise RuntimeError(f"command failed with exit code {result.returncode}: {command_text}\nstdout:\n{result.stdout}\nstderr:\n{result.stderr}")
    return result


def cdt_binary_path(root: Path) -> Path:
    configured = os.environ.get("CDT_BINARY")
    if configured is not None:
        path = Path(configured).expanduser().resolve()
        if path.is_file():
            return path
        message = f"CDT_BINARY does not point to a file: {path}"
        raise FileNotFoundError(message)

    installed = shutil.which("cdt")
    if installed is not None:
        return Path(installed).resolve()

    binary = root / "target" / "release" / "cdt"
    if not binary.is_file():
        run_command(["cargo", "build", "--release", "--bin", "cdt"], cwd=root, timeout=600)
    if not binary.is_file():
        message = f"cdt binary was not produced at {binary}"
        raise FileNotFoundError(message)
    return binary


def read_json(path: Path) -> dict[str, Any]:
    if not path.is_file():
        message = f"summary JSON not found: {path}"
        raise FileNotFoundError(message)
    loaded = json.loads(path.read_text(encoding=UTF8))
    if not isinstance(loaded, dict):
        message = f"summary JSON should be an object: {path}"
        raise TypeError(message)
    return loaded


ROOT = find_repo_root(Path.cwd().resolve())
RUN_PATHS = RunPaths(
    output_dir=ROOT / "target" / "notebooks" / "visualization",
    trace_csv=ROOT / "target" / "notebooks" / "visualization" / "trace.csv",
    summary_json=ROOT / "target" / "notebooks" / "visualization" / "summary.json",
    figure_png=ROOT / "target" / "notebooks" / "visualization" / "cdt_spacetime.png",
    figure_svg=ROOT / "target" / "notebooks" / "visualization" / "cdt_spacetime.svg",
)
RUN_PATHS.output_dir.mkdir(parents=True, exist_ok=True)
CDT_BINARY = cdt_binary_path(ROOT)
print(f"Using cdt binary: {CDT_BINARY}")

## 2. Run a small deterministic simulation

The run is intentionally modest so the notebook works as an example on laptops, CI, and Open OnDemand-backed Jupyter sessions. The cosmological constant is tuned for a short unfixed-volume run, not a production ensemble.

In [ ]:
SIMULATION_PARAMETERS = {
    "dimension": 2,
    "vertices_per_slice": 24,
    "timeslices": 7,
    "topology": "open-boundary",
    "cosmological_constant": 0.9,
    "steps": 160,
    "thermalization_steps": 20,
    "measurement_frequency": 20,
    "seed": 20260612,
}

command = [
    str(CDT_BINARY),
    "--dimension",
    str(SIMULATION_PARAMETERS["dimension"]),
    "--vertices-per-slice",
    str(SIMULATION_PARAMETERS["vertices_per_slice"]),
    "--timeslices",
    str(SIMULATION_PARAMETERS["timeslices"]),
    "--topology",
    str(SIMULATION_PARAMETERS["topology"]),
    "--cosmological-constant",
    str(SIMULATION_PARAMETERS["cosmological_constant"]),
    "--steps",
    str(SIMULATION_PARAMETERS["steps"]),
    "--thermalization-steps",
    str(SIMULATION_PARAMETERS["thermalization_steps"]),
    "--measurement-frequency",
    str(SIMULATION_PARAMETERS["measurement_frequency"]),
    "--seed",
    str(SIMULATION_PARAMETERS["seed"]),
    "--simulate",
    "--output-csv",
    str(RUN_PATHS.trace_csv),
    "--output-json",
    str(RUN_PATHS.summary_json),
]

result = run_command(command, cwd=ROOT, timeout=180)
print(result.stdout.strip())



## 3. Load trace data and choose a volume profile

The trace is a rectangular CSV suitable for Polars. The summary carries measurement records with per-slice volume profiles. The visualization uses the latest recorded profile when available, then falls back to the final trace row or the initial regular profile.

In [ ]:
def read_trace(path: Path) -> pl.DataFrame:
    if not path.is_file():
        message = f"trace CSV not found: {path}"
        raise FileNotFoundError(message)
    trace = pl.read_csv(path).with_columns(
        pl.col("accepted").cast(pl.Boolean),
        pl.col("proposed").cast(pl.Boolean),
    )
    required_columns = {"step", "accepted", "action", "vertices", "triangles"}
    missing = required_columns.difference(trace.columns)
    if missing:
        message = f"trace CSV is missing required columns: {sorted(missing)}"
        raise ValueError(message)
    if trace.height == 0:
        message = f"trace CSV is empty: {path}"
        raise ValueError(message)
    return trace


def latest_volume_profile(summary: dict[str, Any], trace: pl.DataFrame) -> list[int]:
    measurements = summary.get("measurements")
    if isinstance(measurements, list) and measurements:
        candidate = measurements[-1]
        if isinstance(candidate, dict):
            profile = candidate.get("volume_profile")
            if isinstance(profile, list) and profile:
                return [max(3, int(value)) for value in profile]

    profile_columns = sorted(
        [column for column in trace.columns if column.startswith("volume_profile_")],
        key=lambda column: int(column.rsplit("_", maxsplit=1)[-1]),
    )
    if profile_columns:
        last = trace.select(profile_columns).tail(1).row(0)
        profile = [max(3, value) for value in last if value > 0]
        if profile:
            return profile

    config = summary.get("config")
    if not isinstance(config, dict):
        message = "summary config should be an object"
        raise TypeError(message)
    timeslices = int(config.get("timeslices", SIMULATION_PARAMETERS["timeslices"]))
    vertices_per_slice = int(config.get("vertices_per_slice", SIMULATION_PARAMETERS["vertices_per_slice"]))
    return [max(3, vertices_per_slice)] * timeslices


trace = read_trace(RUN_PATHS.trace_csv)
summary = read_json(RUN_PATHS.summary_json)
profile = latest_volume_profile(summary, trace)

trace.select(
    pl.len().alias("rows"),
    pl.col("accepted").mean().alias("acceptance_rate"),
    pl.col("action").mean().alias("mean_action"),
    pl.col("vertices").last().alias("final_vertices"),
    pl.col("triangles").last().alias("final_triangles"),
)



## 4. Render the final CDT spacetime strip

Each horizontal row is a spatial slice and adjacent rows are connected by up/down CDT triangles from the exported final mesh. The vertical plotting coordinate is the exported foliation label, not the raw Delaunay embedding coordinate, so the picture displays causal time directly. The action and volume panels are intentionally diagnostic; suspiciously flat traces are evidence to investigate with `02_analysis_caches.ipynb`, not something to hide.

In [ ]:
def vertex_position(time_index: int, spatial_index: int, slice_size: int, max_slice_size: int) -> tuple[float, float]:
    phase = 2.0 * math.pi * spatial_index / slice_size
    centered_x = spatial_index - (slice_size - 1.0) / 2.0
    spread = max_slice_size / max(slice_size, 1)
    x = centered_x * spread + 0.16 * math.sin(phase + 0.75 * time_index)
    y = float(time_index)
    return (x, y)


def add_edge(edges: set[tuple[int, int]], first: int, second: int) -> None:
    if first != second:
        edge = (first, second) if first < second else (second, first)
        edges.add(edge)


def build_mesh(volume_profile: list[int]) -> Mesh:
    if len(volume_profile) < 2:
        message = "a CDT strip visualization needs at least two time slices"
        raise ValueError(message)
    if min(volume_profile) < 3:
        message = f"each spatial slice needs at least three vertices: {volume_profile}"
        raise ValueError(message)

    max_slice_size = max(volume_profile)
    points: list[tuple[float, float]] = []
    times: list[int] = []
    vertex_ids: list[list[int]] = []
    for time_index, slice_size in enumerate(volume_profile):
        row: list[int] = []
        for spatial_index in range(slice_size):
            row.append(len(points))
            points.append(vertex_position(time_index, spatial_index, slice_size, max_slice_size))
            times.append(time_index)
        vertex_ids.append(row)

    triangles: list[tuple[int, int, int, str]] = []
    for time_index in range(len(volume_profile) - 1):
        lower = vertex_ids[time_index]
        upper = vertex_ids[time_index + 1]
        cells = max(len(lower), len(upper))
        for cell_index in range(cells):
            lower_left = lower[(cell_index * len(lower)) // cells]
            lower_right = lower[((cell_index + 1) * len(lower)) // cells % len(lower)]
            upper_left = upper[(cell_index * len(upper)) // cells]
            upper_right = upper[((cell_index + 1) * len(upper)) // cells % len(upper)]
            if lower_left != lower_right:
                triangles.append((lower_left, lower_right, upper_left, "up"))
            if upper_left != upper_right:
                triangles.append((lower_right, upper_right, upper_left, "down"))

    spacelike: set[tuple[int, int]] = set()
    timelike: set[tuple[int, int]] = set()
    for row in vertex_ids:
        for spatial_index, first in enumerate(row):
            add_edge(spacelike, first, row[(spatial_index + 1) % len(row)])
    for first, second, third, _kind in triangles:
        for edge_first, edge_second in ((first, second), (second, third), (third, first)):
            target = spacelike if times[edge_first] == times[edge_second] else timelike
            add_edge(target, edge_first, edge_second)

    return Mesh(
        points=points,
        times=times,
        triangles=triangles,
        spacelike_edges=sorted(spacelike),
        timelike_edges=sorted(timelike),
    )


def triangle_kind(times: list[int], triangle: tuple[int, int, int]) -> str:
    labels = [times[index] for index in triangle]
    lower = min(labels)
    return "up" if labels.count(lower) == 2 else "down"


def exported_mesh_payload(summary: dict[str, Any]) -> tuple[list[Any], list[Any]] | None:
    final = summary.get("final_triangulation")
    if not isinstance(final, dict):
        return None
    exported_mesh = final.get("mesh")
    if not isinstance(exported_mesh, dict):
        return None
    exported_vertices = exported_mesh.get("vertices")
    exported_triangles = exported_mesh.get("triangles")
    if not isinstance(exported_vertices, list) or not isinstance(exported_triangles, list):
        return None
    return exported_vertices, exported_triangles


def parse_vertex_time(index: int, raw_time: Any, coordinates: list[Any]) -> int:
    if raw_time is None:
        return round(float(coordinates[1]))
    if isinstance(raw_time, bool):
        message = f"exported mesh vertex {index} has non-numeric time label: {raw_time}"
        raise TypeError(message)
    if isinstance(raw_time, int):
        return int(raw_time)
    if isinstance(raw_time, float) and raw_time.is_integer():
        return int(raw_time)
    message = f"exported mesh vertex {index} has non-numeric time label: {raw_time}"
    raise TypeError(message)


if parse_vertex_time(0, 2.0, [0.0, 99.0]) != 2:
    message = "expected numeric time labels to parse as integers"
    raise AssertionError(message)
if parse_vertex_time(1, None, [0.0, 3.0]) != 3:
    message = "expected missing time labels to fall back to rounded y coordinates"
    raise AssertionError(message)
for bad_time in ("2", True, 2.5):
    try:
        parse_vertex_time(2, bad_time, [0.0, 2.0])
    except TypeError as error:
        nonnumeric_error = str(error)
    else:
        nonnumeric_error = ""
    if "non-numeric time label" not in nonnumeric_error:
        message = "expected non-numeric time labels to fail with a clear diagnostic"
        raise AssertionError(message)


def mesh_from_summary(summary: dict[str, Any]) -> Mesh | None:
    payload = exported_mesh_payload(summary)
    if payload is None:
        return None
    exported_vertices, exported_triangles = payload

    points: list[tuple[float, float]] = [(0.0, 0.0)] * len(exported_vertices)
    times: list[int] = [0] * len(exported_vertices)
    for vertex in exported_vertices:
        if not isinstance(vertex, dict):
            message = "exported mesh vertex should be an object"
            raise TypeError(message)
        index = int(vertex["index"])
        coordinates = vertex.get("coordinates")
        if not isinstance(coordinates, list) or len(coordinates) < 2:
            message = f"exported mesh vertex {index} needs at least two coordinates"
            raise ValueError(message)
        if index < 0 or index >= len(points):
            message = f"exported mesh vertex index is out of range: {index}"
            raise ValueError(message)
        time_label = parse_vertex_time(index, vertex.get("time"), coordinates)
        points[index] = (float(coordinates[0]), float(time_label))
        times[index] = time_label

    triangles: list[tuple[int, int, int, str]] = []
    spacelike: set[tuple[int, int]] = set()
    timelike: set[tuple[int, int]] = set()
    for raw_triangle in exported_triangles:
        if not isinstance(raw_triangle, list) or len(raw_triangle) != 3:
            message = f"exported mesh triangle should have exactly three vertices: {raw_triangle}"
            raise ValueError(message)
        triangle = (int(raw_triangle[0]), int(raw_triangle[1]), int(raw_triangle[2]))
        if any(index < 0 or index >= len(points) for index in triangle):
            message = f"exported mesh triangle uses an out-of-range vertex index: {triangle}"
            raise ValueError(message)
        triangles.append((*triangle, triangle_kind(times, triangle)))
        for first, second in ((triangle[0], triangle[1]), (triangle[1], triangle[2]), (triangle[2], triangle[0])):
            target = spacelike if times[first] == times[second] else timelike
            add_edge(target, first, second)

    return Mesh(
        points=points,
        times=times,
        triangles=triangles,
        spacelike_edges=sorted(spacelike),
        timelike_edges=sorted(timelike),
    )


def scaled_for_display(mesh: Mesh) -> Mesh:
    x_values = [point[0] for point in mesh.points]
    y_values = [point[1] for point in mesh.points]
    x_span = max(x_values) - min(x_values)
    y_span = max(y_values) - min(y_values)
    if x_span <= 0.0 or y_span <= 0.0:
        return mesh
    midpoint = (max(x_values) + min(x_values)) / 2.0
    target_x_span = 1.75 * y_span
    scale = max(1.0, min(18.0, target_x_span / x_span))
    scaled_points = [((x - midpoint) * scale, y) for x, y in mesh.points]
    return Mesh(
        points=scaled_points,
        times=mesh.times,
        triangles=mesh.triangles,
        spacelike_edges=mesh.spacelike_edges,
        timelike_edges=mesh.timelike_edges,
    )


def edge_segments(mesh: Mesh, edges: list[tuple[int, int]]) -> list[tuple[tuple[float, float], tuple[float, float]]]:
    return [(mesh.points[first], mesh.points[second]) for first, second in edges]


def triangle_polygons(mesh: Mesh, kind: str) -> list[list[tuple[float, float]]]:
    return [[mesh.points[first], mesh.points[second], mesh.points[third]] for first, second, third, triangle_kind in mesh.triangles if triangle_kind == kind]


def scalar_from_trace(trace: pl.DataFrame, column: str) -> list[float]:
    return [float(value) for value in trace.get_column(column).to_list()]


def render_spacetime(mesh: Mesh, trace: pl.DataFrame, summary: dict[str, Any], paths: RunPaths) -> tuple[Path, Path]:
    aggregate = summary.get("aggregate", {})
    final = summary.get("final_triangulation", {})
    acceptance = float(trace.select(pl.col("accepted").mean()).item())
    final_vertices = int(final.get("vertices", trace.get_column("vertices")[-1]))
    final_triangles = int(final.get("triangles", trace.get_column("triangles")[-1]))
    average_action = aggregate.get("average_action", trace.select(pl.col("action").mean()).item())

    fig = plt.figure(figsize=(14, 8), dpi=200, facecolor="#070912")
    grid = fig.add_gridspec(2, 5, width_ratios=[1.3, 1.3, 1.3, 1.3, 1.0], wspace=0.32, hspace=0.34)
    mesh_axis = fig.add_subplot(grid[:, :4])
    action_axis = fig.add_subplot(grid[0, 4])
    volume_axis = fig.add_subplot(grid[1, 4])

    for axis in (mesh_axis, action_axis, volume_axis):
        axis.set_facecolor("#070912")
        axis.tick_params(colors="#c8d3ea", labelsize=8)
        for spine in axis.spines.values():
            spine.set_color("#2a3558")

    mesh_axis.add_collection(PolyCollection(triangle_polygons(mesh, "up"), facecolors="#39c5d9", edgecolors="#77f7ff", linewidths=0.35, alpha=0.24))
    mesh_axis.add_collection(PolyCollection(triangle_polygons(mesh, "down"), facecolors="#ff7b45", edgecolors="#ffc06d", linewidths=0.35, alpha=0.24))
    mesh_axis.add_collection(LineCollection(edge_segments(mesh, mesh.timelike_edges), colors="#89a8ff", linewidths=0.55, alpha=0.54))
    mesh_axis.add_collection(LineCollection(edge_segments(mesh, mesh.spacelike_edges), colors="#ffe28a", linewidths=0.72, alpha=0.72))

    x_values = [point[0] for point in mesh.points]
    y_values = [point[1] for point in mesh.points]
    mesh_axis.scatter(x_values, y_values, c=mesh.times, cmap="viridis", s=18, edgecolors="#f5f7ff", linewidths=0.25, zorder=4)
    mesh_axis.set_aspect("equal", adjustable="datalim")
    mesh_axis.margins(x=0.08, y=0.06)
    mesh_axis.set_axis_off()
    mesh_axis.set_title("1+1 CDT spacetime triangulation", color="#f8fbff", fontsize=18, pad=18)
    config = summary.get("config", {})
    topology = config.get("topology", SIMULATION_PARAMETERS["topology"]) if isinstance(config, dict) else SIMULATION_PARAMETERS["topology"]
    mesh_axis.text(
        0.01,
        0.02,
        f"{topology} run | seed {SIMULATION_PARAMETERS['seed']} | acceptance {acceptance:.2%} | final V={final_vertices}, T={final_triangles}",
        transform=mesh_axis.transAxes,
        color="#dbe8ff",
        fontsize=9,
        alpha=0.9,
    )

    steps = scalar_from_trace(trace, "step")
    action_axis.plot(steps, scalar_from_trace(trace, "action"), color="#77f7ff", linewidth=1.7)
    action_axis.set_title("Action", color="#f8fbff", fontsize=11)
    action_axis.set_xlabel("step", color="#c8d3ea", fontsize=8)
    action_axis.grid(color="#233052", alpha=0.5, linewidth=0.6)

    volume_axis.plot(steps, scalar_from_trace(trace, "vertices"), color="#ffc06d", linewidth=1.7, label="vertices")
    volume_axis.plot(steps, scalar_from_trace(trace, "triangles"), color="#c79bff", linewidth=1.2, label="triangles")
    volume_axis.set_title("Volume", color="#f8fbff", fontsize=11)
    volume_axis.set_xlabel("step", color="#c8d3ea", fontsize=8)
    volume_axis.legend(loc="best", fontsize=7, frameon=False, labelcolor="#dbe8ff")
    volume_axis.grid(color="#233052", alpha=0.5, linewidth=0.6)

    fig.suptitle(f"causal-triangulations example visualization | mean action {float(average_action):.3f}", color="#b8c7ff", fontsize=10, y=0.98)
    paths.figure_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(paths.figure_png, bbox_inches="tight", facecolor=fig.get_facecolor())
    fig.savefig(paths.figure_svg, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()
    return paths.figure_png, paths.figure_svg


mesh = mesh_from_summary(summary) or build_mesh(profile)
render_spacetime(scaled_for_display(mesh), trace, summary, RUN_PATHS)

## 5. Use the generated image

The rendered files are written to `target/notebooks/visualization/cdt_spacetime.png` and `target/notebooks/visualization/cdt_spacetime.svg`. Try changing `SIMULATION_PARAMETERS["seed"]`, `SIMULATION_PARAMETERS["cosmological_constant"]`, or the number of time slices to explore different short-run geometries.